In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pickle
import os
import shutil
import numpy as np
import cv2
import glob
from tqdm import tqdm
from scipy.io import loadmat
import re
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [3]:
data_path = 'drive/MyDrive/handwriting data'
project_path = 'drive/MyDrive/Colab Notebooks/handwriting recognition'
author_images_dir = os.path.join(data_path, '1_ImagesRotated')
author_mat_dir = os.path.join(data_path, '5_DataDarkLines')
output_path = os.path.join(project_path, '1_ImagesRotated')

In [4]:
def save_pickle(path: str, data: object) -> None:
  with open(path, 'wb') as f:
    pickle.dump(data, f)

In [5]:
def natural_key(name: str) -> list:
  return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', name)]

In [6]:
def ensure_file_exist(filepath: str) -> None:
  if not os.path.exists(filepath):
    raise FileNotFoundError(f"File not found: {filepath}")

In [7]:
def load_image(image_path: str) -> np.ndarray:
  ensure_file_exist(image_path)
  return cv2.imread(image_path)

In [8]:
def binarize_and_invert(image: np.ndarray) -> np.ndarray:
  # Convert to grayscale if needed
  if len(image.shape) == 3:
    image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
  # Apply Otsu binarization and invert so letters are white
  _, binary = cv2.threshold(image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
  binary = cv2.bitwise_not(binary)
  return binary

In [9]:
def is_mostly_black(patch: np.ndarray, threshold: float) -> bool:
  black_pixels = np.sum(patch == 0)
  ratio = black_pixels / patch.size
  return ratio >= threshold

In [10]:
def save_patches(
    patches: list[np.ndarray],
    output_dir: str,
    prefix: str,
    start_id: int = 0
    ) -> list[str]:
  filenames: list[str] = []

  for i, patch in enumerate(patches):
    fname = f"{prefix}_patch_{start_id + i:03d}.npy"
    fpath = os.path.join(output_dir, fname)
    np.save(fpath, patch)
    filenames.append(fname)

  return filenames

In [11]:
def normalize_line_height(
    start_y: int,
    end_y: int,
    target_height: int = 180
    ) -> tuple[int, int]:
  current_height = end_y - start_y
  if current_height < target_height:
    diff = target_height - current_height
    pad_top = diff // 2
    pad_bottom = diff - pad_top
    return start_y - pad_top, end_y + pad_bottom
  else:
    center = (start_y + end_y) // 2
    return center - target_height // 2, center + target_height // 2

In [12]:
class AuthorProcessor:
  def __init__(
    self,
    label_id: int,
    author: str,
    binary_output_dir: str,
    author_images_dir: str,
    author_mat_dir: str,
    patch_size: tuple[int, int] = (180, 160),
    stride_x: int = 30,
    stride_y: int = 10,
    threshold: float = 0.95,
    val_size: float = 0.15
  ) -> None:
    self.label_id = label_id
    self.author = author
    self.binary_output_dir = binary_output_dir
    self.author_images_dir = author_images_dir
    self.author_mat_dir = author_mat_dir
    self.patch_size = patch_size
    self.stride_x = stride_x
    self.stride_y = stride_y
    self.threshold = threshold
    self.val_size = val_size

    self.image: np.ndarray = self._load_image()
    self.lines_y, self.top_test_area, self.bottom_test_area = self._load_mat()

  def _load_image(self) -> np.ndarray:
    path_image = os.path.join(self.author_images_dir, f"{self.author}.jpg")
    image: np.ndarray = load_image(path_image)
    return binarize_and_invert(image.copy())

  def _load_mat(self) -> tuple[list[tuple[int, int]], int, int]:
    path_mat = os.path.join(self.author_mat_dir, f"{self.author}.mat")
    ensure_file_exist(path_mat)
    mat = loadmat(path_mat)

    peaks: np.ndarray = mat['peaks_indices'].flatten()
    scale: float = float(mat['SCALE_FACTOR'].flatten()[0])
    top: int = int(mat['top_test_area'].flatten()[0])
    bottom: int = int(mat['bottom_test_area'].flatten()[0])

    peaks_scaled = [int(p * scale) for p in peaks]
    lines_y = [(peaks_scaled[i], peaks_scaled[i + 1]) for i in range(len(peaks_scaled) - 1)]
    return lines_y, top, bottom

  def _extract_patches_from_segment(self, segment: np.ndarray) -> list[np.ndarray]:
    patches: list[np.ndarray] = []

    for x in range(0, segment.shape[1] - self.patch_size[1] + 1, self.stride_x):
      patch = segment[:, x:x + self.patch_size[1]]
      if not is_mostly_black(patch, self.threshold):
        patches.append(patch)

    return patches

  def _process_test_area(self) -> list[str]:
    start, end = self.top_test_area, self.bottom_test_area
    test_segment = self.image[start:end, :]
    patches = self._extract_patches_from_segment(test_segment)
    filenames = save_patches(patches, self.binary_output_dir, f"{self.author}_test")
    return filenames

  def _process_lines(self) -> dict[int, list[str]]:
    line_to_patch_files:dict[int, list[str]] = {}
    patch_id = 0
    line_idx = 0

    for line_index, (start_y, end_y) in enumerate(self.lines_y):
      start_y, end_y = normalize_line_height(start_y, end_y)
      segment = self.image[start_y:end_y, :]
      if not is_mostly_black(segment, self.threshold):
        patches = self._extract_patches_from_segment(segment)
        filenames = save_patches(patches, self.binary_output_dir, f"{self.author}", patch_id)
        line_to_patch_files[line_idx] = filenames
        patch_id += len(filenames)
        line_idx += 1

    return line_to_patch_files

  def _build_partition(self) -> dict[str, list[str]]:
    test_patches = self._process_test_area()
    line_to_patch_files = self._process_lines()

    partition = {'train': [], 'validation': [], 'test': test_patches}
    all_lines = list(line_to_patch_files.items())

    if len(all_lines) == 1:
      partition['train'].extend(all_lines[0][1])
    elif len(all_lines) == 2:
      partition['train'].extend(all_lines[0][1])
      partition['validation'].extend(all_lines[1][1])
    elif len(all_lines) == 3:
      sorted_lines = sorted(all_lines, key=lambda x: len(x[1]))
      partition['validation'].extend(sorted_lines[0][1])
      partition['train'].extend(sorted_lines[1][1])
      partition['train'].extend(sorted_lines[2][1])
    else:
      total_patches = sum(len(patches) for _, patches in all_lines)
      target_val_patches = int(total_patches * self.val_size)
      val_count = 0

      for _, patches in sorted(all_lines, key=lambda x: len(x[1])):
        if val_count < target_val_patches:
          partition['validation'].extend(patches)
          val_count += len(patches)
        else:
          partition['train'].extend(patches)

    return partition

  def generate(self) -> tuple[dict[str, list[str]], dict[str, int]]:

    partition = self._build_partition()

    labels = dict((fname, self.label_id) for fname in partition['train'] + partition['validation'] + partition['test'])

    return partition, labels

In [13]:
def collect_patches(binary_dir: str, filenames: list[str]) -> dict:
  data = {}
  for filename in filenames:
    filepath = os.path.join(binary_dir, filename)
    data[filename] = np.load(filepath)
  return data

In [14]:
def save_dataset(out_path: str, binary_output_dir: str, amount: int, global_partition: dict, global_labels: dict) -> None:
  save_pickle(os.path.join(out_path, f"partition_data_{amount}.pkl"), global_partition)
  save_pickle(os.path.join(out_path, f"labels_data_{amount}.pkl"), global_labels)

  all_filenames = global_partition["train"] + global_partition["validation"] + global_partition["test"]
  patch_data = collect_patches(binary_output_dir, all_filenames)

  save_pickle(os.path.join(out_path, f"patch_data_{amount}.pkl"), patch_data)

In [15]:
def recreate_path(path: str) -> None:
  if os.path.exists(path):
    shutil.rmtree(path)
  os.makedirs(path)

In [16]:
# List of all the files names
author_images: list = glob.glob(f"{author_images_dir}/*.jpg")

# get only the file name without the extension
author_names: list = [os.path.splitext(os.path.basename(filename))[0]
                for filename in author_images]

authors_names: list[str] = sorted(author_names, key=natural_key)

In [17]:
amounts: list[int] = [100, 150, 180, 204]

In [18]:
for amount in amounts:
  selected_authors: list[str] = authors_names[:amount]
  binary_output_dir = f"binary_patches_{amount}"
  recreate_path(binary_output_dir)

  dir = os.path.join(output_path, f"dataset_{amount}")
  recreate_path(dir)

  global_partition = {
    "train": [],
    "validation": [],
    "test": []
  }
  global_labels = {}
  pbar = tqdm(selected_authors, total=len(selected_authors), desc=f"Processing authors (amount: {amount})")

  for idx, author in enumerate(pbar):
    pbar.set_postfix(current_author=author)
    builder = AuthorProcessor(idx, author, binary_output_dir, author_images_dir, author_mat_dir)
    partition, labels = builder.generate()

    for key in global_partition:
      global_partition[key] += partition[key]
    global_labels.update(labels)

  save_dataset(dir, binary_output_dir, amount, global_partition, global_labels)
  pbar.close()

Processing authors (amount: 204): 100%|██████████| 204/204 [09:37<00:00,  2.83s/it, current_author=lines4_Page_07]
